[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left" alt="Open In Colab"/>](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/02_apigee_llm_gateway.ipynb)

<br><br>


# Apigee AI Gateway — LLM Governance & Policies (AFT)

**Repository Location:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/02_apigee_llm_gateway.ipynb`](https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/02_apigee_llm_gateway.ipynb)  
**Execution Constraint:** This notebook is version-controlled in GitHub and designed to run cleanly when opened directly from that repository path in Google Colab (no local-only paths, no machine-specific assumptions, and no state carried between sessions).

> **Scope Note:** **MCP tool-call validation** and **A2A agent-to-agent governance** are **OUT OF SCOPE** for this notebook — they are implemented in a separate notebook in the same repository. No structure or placeholders are reserved for them here. Summary and cleanup is numbered **Area 10**.

**Framework & Stability Rule:**
- **Toolkit:** **AFT (`aft` — Apigee Foundation Toolkit / Apigee Feature Templater)** (`https://github.com/apigee/apigee-templater`)
- **Base Repository:** [`https://github.com/gcp-samples/apigee-template-repository`](https://github.com/gcp-samples/apigee-template-repository)
- **Allowed Tiers Only:** Uses **ONLY** items published under **`## Stable Templates`** and **`## Stable Features`** in the base repository. Everything under **Drafts** (`## Draft Templates`, `## Draft Extension Features`, `features/draft/*`) is strictly excluded.
- **Assumptions:** Apigee X is already provisioned with `APIGEE_ENV` attached to an environment group, and target LLM backends are reachable.

---

## Step 0 — Preflight (Blocking Gate)
Run every cell in **Step 0 (`0.1`–`0.8`)** in order before running any later section. Each check is a hard gate that verifies environment readiness, effective IAM permissions, enabled GCP APIs, live Gemini connectivity, and the AFT CLI.

In [ ]:
#@title 0.1 Inputs & Global Configuration
# Single input surface for Step 0. Collects and validates all project, Apigee, service account,
# Gemini, and behaviour parameters up front before any cell makes an API call.
import os
import re

#@markdown ### Project & Apigee
PROJECT_ID = "" #@param {type:"string"}
APIGEE_ORG = "" #@param {type:"string"}
APIGEE_ENV = "dev" #@param {type:"string"}
ENV_GROUP = "dev-group" #@param {type:"string"}
REGION = "europe-west1" #@param {type:"string"}
HOSTNAME = "" #@param {type:"string"}

#@markdown ### Service Account
SERVICE_ACCOUNT = "" #@param {type:"string"}
SA_NAME = "apigee-ai-gateway-sa" #@param {type:"string"}
CREATE_SERVICE_ACCOUNT = True #@param {type:"boolean"}

#@markdown ### Gemini
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

#@markdown ### Behaviour Flags
ENABLE_APIS = False #@param {type:"boolean"}

PROJECT_ID = PROJECT_ID.strip()
APIGEE_ORG = (APIGEE_ORG.strip() or PROJECT_ID)
APIGEE_ENV = APIGEE_ENV.strip()
ENV_GROUP = ENV_GROUP.strip()
REGION = REGION.strip()
HOSTNAME = HOSTNAME.strip()
SERVICE_ACCOUNT = SERVICE_ACCOUNT.strip()
SA_NAME = SA_NAME.strip()
GEMINI_MODEL = GEMINI_MODEL.strip().removeprefix("google/")

# 1. Validate required fields are non-empty
missing = [
    k for k, v in {
        "PROJECT_ID": PROJECT_ID,
        "APIGEE_ENV": APIGEE_ENV,
        "REGION": REGION,
        "GEMINI_MODEL": GEMINI_MODEL,
    }.items()
    if not v
]
if missing:
    raise SystemExit(f"❌ HALT: Missing required parameter(s): {', '.join(missing)}. Fill them in Cell 0.1 and re-run.")

# 2. Validate SERVICE_ACCOUNT format if supplied
if SERVICE_ACCOUNT and not re.fullmatch(r"[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.gserviceaccount\.com", SERVICE_ACCOUNT):
    raise SystemExit(
        f"❌ HALT: Invalid SERVICE_ACCOUNT '{SERVICE_ACCOUNT}'. "
        "Expected a valid Google Cloud service account email ending in '.gserviceaccount.com'."
    )

# 3. Validate SA_NAME is a valid GCP service account ID (lowercase, 6-30 chars, hyphens)
if not re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", SA_NAME):
    raise SystemExit(
        f"❌ HALT: Invalid SA_NAME '{SA_NAME}'. "
        "Service account ID must be 6–30 characters of lowercase letters, digits, and hyphens (starting with a letter)."
    )

# 4. Reject impossible combination: no existing SERVICE_ACCOUNT and CREATE_SERVICE_ACCOUNT=False
if not SERVICE_ACCOUNT and not CREATE_SERVICE_ACCOUNT:
    raise SystemExit(
        "❌ HALT: SERVICE_ACCOUNT is empty and CREATE_SERVICE_ACCOUNT is False. "
        "Either provide an existing SERVICE_ACCOUNT email or set CREATE_SERVICE_ACCOUNT=True."
    )

# Export standard environment variables consumed by AFT and Apigee templates (env.sh convention)
for k, v in {
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "APIGEE_ORG": APIGEE_ORG,
    "APIGEE_ENV": APIGEE_ENV,
    "ENV_GROUP": ENV_GROUP,
    "REGION": REGION,
    "HOSTNAME": HOSTNAME,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT,
    "SA_NAME": SA_NAME,
    "GEMINI_MODEL": GEMINI_MODEL,
}.items():
    os.environ[k] = v

PREFLIGHT = {
    "inputs": True,
    "auth": False,
    "apigee_org_env": False,
    "iam": False,
    "service_account": False,
    "apigee_apis": False,
    "gemini": False,
    "aft": False,
}

sa_summary = SERVICE_ACCOUNT if SERVICE_ACCOUNT else f"{SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com (create={CREATE_SERVICE_ACCOUNT})"
print("=" * 72)
print("✅ Inputs validated and recorded:")
print(f"  PROJECT_ID             : {PROJECT_ID}")
print(f"  APIGEE_ORG             : {APIGEE_ORG}")
print(f"  APIGEE_ENV / ENV_GROUP : {APIGEE_ENV} / {ENV_GROUP or '(auto-detect)'}")
print(f"  REGION                 : {REGION}")
print(f"  HOSTNAME               : {HOSTNAME or '(auto-detect from envgroup)'}")
print(f"  SERVICE_ACCOUNT        : {sa_summary}")
print(f"  GEMINI_MODEL           : {GEMINI_MODEL}")
print(f"  ENABLE_APIS            : {ENABLE_APIS}")
print("=" * 72)

In [ ]:
#@title 0.2 Authenticate Caller & Print Active Identity
# Authenticates the Colab session with Google Cloud ADC and initializes an AuthorizedSession
# wrapper that returns (http_status, response_dict) instead of raising unhandled stack traces.
import subprocess
import google.auth
from google.auth.transport.requests import AuthorizedSession, Request

try:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user(project_id=PROJECT_ID)
except ImportError:
    pass

subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=True, capture_output=True)

try:
    CREDENTIALS, _ = google.auth.default(
        scopes=[
            "https://www.googleapis.com/auth/cloud-platform",
            "https://www.googleapis.com/auth/userinfo.email",
            "openid",
        ]
    )
    CREDENTIALS.refresh(Request())
    AUTHSESSION = AuthorizedSession(CREDENTIALS)
except Exception as exc:
    raise SystemExit(f"❌ HALT: Failed to initialize Google Cloud credentials: {exc}")

def gcp_request(method: str, url: str, json_body=None, headers=None):
    """Executes an authenticated GCP REST call and always returns (status_code, parsed_body)."""
    try:
        resp = AUTHSESSION.request(method, url, json=json_body, headers=headers, timeout=30)
    except Exception as exc:
        return 0, {"error": str(exc), "url": url}
    try:
        data = resp.json()
    except Exception:
        data = {"raw": resp.text}
    return resp.status_code, data

_, token_info = gcp_request("GET", f"https://oauth2.googleapis.com/tokeninfo?access_token={CREDENTIALS.token}")
ACTIVE_IDENTITY = (
    token_info.get("email")
    or getattr(CREDENTIALS, "service_account_email", None)
    or subprocess.run(["gcloud", "config", "get-value", "account"], capture_output=True, text=True).stdout.strip()
)

if not ACTIVE_IDENTITY:
    raise SystemExit("❌ HALT: Could not resolve active caller identity.")

PREFLIGHT["auth"] = True
print(f"✅ Authenticated identity: {ACTIVE_IDENTITY}")

In [ ]:
#@title 0.3 Validate Apigee Organization, Environment & Environment Group
# In Apigee X, API proxies deploy to an Environment (e.g., 'eval'), while external north-bound
# hostnames are defined on an Environment Group that attaches one or more Environments.
APIGEE_BASE = f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}"

status, org_data = gcp_request("GET", APIGEE_BASE)
if status != 200:
    raise SystemExit(f"❌ HALT: Apigee organization '{APIGEE_ORG}' not found or inaccessible (HTTP {status}): {org_data}")

org_state = org_data.get("state", "UNKNOWN")
if org_state != "ACTIVE":
    raise SystemExit(f"❌ HALT: Apigee organization '{APIGEE_ORG}' state is '{org_state}' (expected 'ACTIVE').")

status, env_data = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}")
if status != 200:
    raise SystemExit(f"❌ HALT: Apigee environment '{APIGEE_ENV}' not found in org '{APIGEE_ORG}' (HTTP {status}): {env_data}")

status, eg_list = gcp_request("GET", f"{APIGEE_BASE}/envgroups")
envgroups = eg_list.get("environmentGroups", []) if status == 200 else []
if not envgroups:
    raise SystemExit(f"❌ HALT: No environment groups found in org '{APIGEE_ORG}' (HTTP {status}): {eg_list}")

matched_group, matched_hosts = None, []
for eg in envgroups:
    eg_name = eg["name"]
    if ENV_GROUP and eg_name != ENV_GROUP:
        continue
    att_status, att_data = gcp_request("GET", f"{APIGEE_BASE}/envgroups/{eg_name}/attachments")
    if att_status != 200:
        raise SystemExit(f"❌ HALT: Failed to read attachments for envgroup '{eg_name}' (HTTP {att_status}): {att_data}")
    attached_envs = [a.get("environment") for a in att_data.get("environmentGroupAttachments", [])]
    if APIGEE_ENV in attached_envs:
        matched_group = eg_name
        matched_hosts = eg.get("hostnames", [])
        break

if not matched_group:
    target_msg = f"envgroup '{ENV_GROUP}'" if ENV_GROUP else "any environment group"
    raise SystemExit(f"❌ HALT: Environment '{APIGEE_ENV}' is not attached to {target_msg} in org '{APIGEE_ORG}'.")

ENV_GROUP = matched_group
if not HOSTNAME:
    if not matched_hosts:
        raise SystemExit(f"❌ HALT: Environment group '{ENV_GROUP}' has no configured hostnames.")
    HOSTNAME = matched_hosts[0]
elif HOSTNAME not in matched_hosts:
    print(f"⚠️ Warning: HOSTNAME '{HOSTNAME}' not listed in envgroup '{ENV_GROUP}' hostnames {matched_hosts}.")

os.environ["ENV_GROUP"] = ENV_GROUP
os.environ["HOSTNAME"] = HOSTNAME
PREFLIGHT["apigee_org_env"] = True

print(f"✅ Apigee Org:   {APIGEE_ORG} (state={org_state})")
print(f"✅ Apigee Env:   {APIGEE_ENV} (attached to envgroup='{ENV_GROUP}')")
print(f"✅ Gateway Host: {HOSTNAME}")

In [ ]:
#@title 0.4 Verify Effective Apigee IAM Permissions (testIamPermissions)
# Tests actual effective IAM permissions (rather than inspecting role bindings, which can miss
# inherited policies or custom roles) required to manage proxies, revisions, deployments,
# shared flows, KVMs, and target servers.
REQUIRED_APIGEE_PERMISSIONS = [
    # (Domain, Permission, Least-Privilege Role)
    ("Proxies",        "apigee.proxies.create",              "roles/apigee.apiAdminV2"),
    ("Proxies",        "apigee.proxies.get",                 "roles/apigee.apiAdminV2"),
    ("Proxies",        "apigee.proxies.update",              "roles/apigee.apiAdminV2"),
    ("Proxies",        "apigee.proxies.delete",              "roles/apigee.apiAdminV2"),
    ("Revisions",      "apigee.proxyrevisions.get",          "roles/apigee.apiAdminV2"),
    ("Revisions",      "apigee.proxyrevisions.update",       "roles/apigee.apiAdminV2"),
    ("Revisions",      "apigee.proxyrevisions.delete",       "roles/apigee.apiAdminV2"),
    ("Revisions",      "apigee.proxyrevisions.deploy",       "roles/apigee.apiAdminV2"),
    ("Revisions",      "apigee.proxyrevisions.undeploy",     "roles/apigee.apiAdminV2"),
    ("Deployments",    "apigee.deployments.create",          "roles/apigee.environmentAdmin"),
    ("Deployments",    "apigee.deployments.get",             "roles/apigee.environmentAdmin"),
    ("Deployments",    "apigee.deployments.update",          "roles/apigee.environmentAdmin"),
    ("Deployments",    "apigee.deployments.delete",          "roles/apigee.environmentAdmin"),
    ("Shared Flows",   "apigee.sharedflows.create",          "roles/apigee.apiAdminV2"),
    ("Shared Flows",   "apigee.sharedflows.get",             "roles/apigee.apiAdminV2"),
    ("Shared Flows",   "apigee.sharedflows.delete",          "roles/apigee.apiAdminV2"),
    ("Shared Flows",   "apigee.sharedflowrevisions.update",  "roles/apigee.apiAdminV2"),
    ("Shared Flows",   "apigee.sharedflowrevisions.deploy",  "roles/apigee.apiAdminV2"),
    ("Shared Flows",   "apigee.sharedflowrevisions.undeploy","roles/apigee.apiAdminV2"),
    ("KVMs",           "apigee.keyvaluemaps.create",         "roles/apigee.environmentAdmin"),
    ("KVMs",           "apigee.keyvaluemaps.get",            "roles/apigee.environmentAdmin"),
    ("KVMs",           "apigee.keyvaluemaps.update",         "roles/apigee.environmentAdmin"),
    ("KVMs",           "apigee.keyvaluemaps.delete",         "roles/apigee.environmentAdmin"),
    ("KVM Entries",    "apigee.keyvaluemapentries.create",   "roles/apigee.environmentAdmin"),
    ("KVM Entries",    "apigee.keyvaluemapentries.get",      "roles/apigee.environmentAdmin"),
    ("KVM Entries",    "apigee.keyvaluemapentries.update",   "roles/apigee.environmentAdmin"),
    ("KVM Entries",    "apigee.keyvaluemapentries.delete",   "roles/apigee.environmentAdmin"),
    ("Target Servers", "apigee.targetservers.create",        "roles/apigee.environmentAdmin"),
    ("Target Servers", "apigee.targetservers.get",           "roles/apigee.environmentAdmin"),
    ("Target Servers", "apigee.targetservers.update",        "roles/apigee.environmentAdmin"),
    ("Target Servers", "apigee.targetservers.delete",        "roles/apigee.environmentAdmin"),
]

perm_list = [p for _, p, _ in REQUIRED_APIGEE_PERMISSIONS]
iam_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions"
status, iam_resp = gcp_request("POST", iam_url, {"permissions": perm_list})
if status != 200:
    raise SystemExit(f"❌ HALT: testIamPermissions call failed (HTTP {status}): {iam_resp}")

granted_set = set(iam_resp.get("permissions", []))
missing_roles = set()

print(f"{'DOMAIN':<16} | {'PERMISSION':<36} | {'STATUS':<6} | {'GRANTING ROLE'}")
print("-" * 92)
for domain, perm, role in REQUIRED_APIGEE_PERMISSIONS:
    ok = perm in granted_set
    if not ok:
        missing_roles.add(role)
    print(f"{domain:<16} | {perm:<36} | {'PASS' if ok else 'FAIL':<6} | {role} (or roles/apigee.admin)")

if missing_roles:
    member_prefix = "serviceAccount" if ACTIVE_IDENTITY.endswith(".gserviceaccount.com") else "user"
    print("\n❌ Missing effective Apigee IAM permissions. Run the following command(s) and re-run this cell:\n")
    for role in sorted(missing_roles):
        print(
            f"gcloud projects add-iam-policy-binding {PROJECT_ID} \\\n"
            f"  --member=\"{member_prefix}:{ACTIVE_IDENTITY}\" \\\n"
            f"  --role=\"{role}\""
        )
    print(
        f"\n# Or grant full Apigee Admin:\n"
        f"gcloud projects add-iam-policy-binding {PROJECT_ID} \\\n"
        f"  --member=\"{member_prefix}:{ACTIVE_IDENTITY}\" \\\n"
        f"  --role=\"roles/apigee.admin\""
    )
    raise SystemExit("❌ HALT: IAM verification failed.")

PREFLIGHT["iam"] = True
print("\n✅ All required Apigee permissions verified.")

In [ ]:
#@title 0.4b Service Account Provisioning
# Ensures a dedicated, least-privilege service account exists for Apigee AI Gateway proxies
# to authenticate to AI backends, consuming SERVICE_ACCOUNT, SA_NAME, and CREATE_SERVICE_ACCOUNT from 0.1.
import time

PREFLIGHT["service_account"] = False
REQUIRED_SA_ROLES = ["roles/aiplatform.user"]

def get_project_sa_roles(sa_email: str) -> list:
    """Returns sorted project IAM roles currently bound to sa_email (with brief retry)."""
    member = f"serviceAccount:{sa_email}"
    for _ in range(4):
        p_status, policy = gcp_request(
            "POST",
            f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:getIamPolicy",
            {},
        )
        if p_status == 200:
            return sorted(
                b.get("role", "")
                for b in policy.get("bindings", [])
                if member in b.get("members", [])
            )
        time.sleep(2)
    raise SystemExit(f"❌ HALT: Failed to read project IAM policy (HTTP {p_status}): {policy}")

def ensure_sa_roles_bound(sa_email: str, required_roles: list) -> list:
    """Idempotently binds required_roles to sa_email and retries checks for IAM propagation."""
    member = f"serviceAccount:{sa_email}"
    for role in required_roles:
        bound_ok = False
        for _ in range(5):
            res = subprocess.run(
                [
                    "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
                    f"--member={member}",
                    f"--role={role}",
                    "--condition=None",
                    "--quiet",
                ],
                capture_output=True,
                text=True,
            )
            if res.returncode == 0:
                bound_ok = True
                break
            time.sleep(2)
        if not bound_ok:
            raise SystemExit(
                f"❌ HALT: Failed to bind role '{role}' to '{sa_email}':\n"
                f"{(res.stderr or res.stdout).strip()}"
            )

    for _ in range(6):
        granted_roles = get_project_sa_roles(sa_email)
        if all(r in granted_roles for r in required_roles):
            return granted_roles
        time.sleep(2)

    raise SystemExit(
        f"❌ HALT: Role binding(s) {required_roles} on '{sa_email}' did not propagate in time. "
        f"Current roles: {granted_roles}"
    )

if SERVICE_ACCOUNT:
    sa_url = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
    sa_status, sa_data = gcp_request("GET", sa_url)
    if sa_status != 200:
        raise SystemExit(
            f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' does not exist or is inaccessible "
            f"(HTTP {sa_status}): {sa_data}"
        )
    if sa_data.get("disabled", False):
        raise SystemExit(
            f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' is disabled. Enable it with:\n"
            f"gcloud iam service-accounts enable {SERVICE_ACCOUNT} --project={PROJECT_ID}"
        )
    SA_STATUS = "REUSED"
    SA_GRANTED_ROLES = get_project_sa_roles(SERVICE_ACCOUNT)
    print(f"ℹ️ Verified provided service account: {SERVICE_ACCOUNT} (enabled)")
else:
    if not SA_NAME:
        raise SystemExit("❌ HALT: SA_NAME cannot be empty when SERVICE_ACCOUNT is not provided.")

    target_sa_email = f"{SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"
    sa_url = f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts/{target_sa_email}"
    sa_status, sa_data = gcp_request("GET", sa_url)

    manual_create_cmd = (
        f"gcloud iam service-accounts create {SA_NAME} \\\n"
        f"  --project={PROJECT_ID} \\\n"
        f"  --display-name=\"Apigee AI Gateway Service Account\" \\\n"
        f"  --description=\"Created by 02_apigee_llm_gateway.ipynb for Apigee AI Gateway proxies\""
    )
    manual_bind_cmds = "\n".join(
        f"gcloud projects add-iam-policy-binding {PROJECT_ID} \\\n"
        f"  --member=\"serviceAccount:{target_sa_email}\" \\\n"
        f"  --role=\"{role}\""
        for role in REQUIRED_SA_ROLES
    )

    if sa_status == 200:
        if sa_data.get("disabled", False):
            raise SystemExit(
                f"❌ HALT: Existing service account '{target_sa_email}' is disabled. Enable it with:\n"
                f"gcloud iam service-accounts enable {target_sa_email} --project={PROJECT_ID}"
            )
        SERVICE_ACCOUNT = target_sa_email
        SA_STATUS = "REUSED"
        print(f"ℹ️ Service account '{SERVICE_ACCOUNT}' already exists — reusing it.")
    elif sa_status == 404:
        if not CREATE_SERVICE_ACCOUNT:
            print(
                f"❌ Service account '{target_sa_email}' does not exist and CREATE_SERVICE_ACCOUNT=False.\n"
                f"Set CREATE_SERVICE_ACCOUNT=True and re-run, or run manually:\n\n"
                f"{manual_create_cmd}\n\n{manual_bind_cmds}"
            )
            raise SystemExit("❌ HALT: Service account is missing and CREATE_SERVICE_ACCOUNT is False.")

        SA_CREATE_PERMS = [
            ("iam.serviceAccounts.create", "roles/iam.serviceAccountAdmin"),
            ("resourcemanager.projects.setIamPolicy", "roles/resourcemanager.projectIamAdmin"),
        ]
        p_status, p_resp = gcp_request(
            "POST",
            f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions",
            {"permissions": [p for p, _ in SA_CREATE_PERMS]},
        )
        if p_status != 200:
            raise SystemExit(f"❌ HALT: testIamPermissions call failed (HTTP {p_status}): {p_resp}")

        granted_create_perms = set(p_resp.get("permissions", []))
        missing_create_perms = [(p, r) for p, r in SA_CREATE_PERMS if p not in granted_create_perms]
        if missing_create_perms:
            print("❌ Caller lacks permission(s) to create and bind the service account:\n")
            for perm, role in missing_create_perms:
                print(f"  - Missing permission: {perm} (granting role: {role})")
            print("\nRun the following commands manually, then re-run this cell:\n")
            print(f"{manual_create_cmd}\n\n{manual_bind_cmds}")
            raise SystemExit("❌ HALT: Insufficient permissions for service account creation.")

        c_status, c_data = gcp_request(
            "POST",
            f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts",
            {
                "accountId": SA_NAME,
                "serviceAccount": {
                    "displayName": "Apigee AI Gateway Service Account",
                    "description": "Created by 02_apigee_llm_gateway.ipynb for Apigee AI Gateway proxies",
                },
            },
        )
        if c_status not in (200, 201):
            raise SystemExit(
                f"❌ HALT: Failed to create service account '{target_sa_email}' "
                f"(HTTP {c_status}): {c_data}"
            )

        for _ in range(5):
            chk_status, _ = gcp_request("GET", sa_url)
            if chk_status == 200:
                break
            time.sleep(2)

        SERVICE_ACCOUNT = c_data.get("email", target_sa_email)
        SA_STATUS = "CREATED"
        print(f"✅ Created service account: {SERVICE_ACCOUNT}")
    else:
        raise SystemExit(
            f"❌ HALT: Failed to check service account '{target_sa_email}' (HTTP {sa_status}): {sa_data}"
        )

    b_status, b_resp = gcp_request(
        "POST",
        f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions",
        {"permissions": ["resourcemanager.projects.setIamPolicy"]},
    )
    if b_status != 200:
        raise SystemExit(f"❌ HALT: testIamPermissions call failed (HTTP {b_status}): {b_resp}")

    if "resourcemanager.projects.setIamPolicy" not in b_resp.get("permissions", []):
        print(
            "❌ Missing permission: resourcemanager.projects.setIamPolicy "
            "(granting role: roles/resourcemanager.projectIamAdmin)\n\n"
            f"Run manually to bind the required role(s) on '{SERVICE_ACCOUNT}':\n\n{manual_bind_cmds}"
        )
        raise SystemExit("❌ HALT: Missing resourcemanager.projects.setIamPolicy permission to bind roles.")

    SA_GRANTED_ROLES = ensure_sa_roles_bound(SERVICE_ACCOUNT, REQUIRED_SA_ROLES)

os.environ["SERVICE_ACCOUNT"] = SERVICE_ACCOUNT
PREFLIGHT["service_account"] = True

print(f"✅ Resolved SERVICE_ACCOUNT: {SERVICE_ACCOUNT} ({SA_STATUS})")
print(f"✅ Granted Project Roles:    {', '.join(SA_GRANTED_ROLES) if SA_GRANTED_ROLES else 'None'}")

In [ ]:
#@title 0.5 Verify Required Apigee APIs
# Checks Service Usage API for core Apigee and Resource Manager APIs without enabling silently.
APIGEE_SERVICES = [
    "apigee.googleapis.com",
    "cloudresourcemanager.googleapis.com",
]

disabled_apigee_svcs = []
print(f"{'SERVICE':<40} | {'STATE'}")
print("-" * 55)
for svc in APIGEE_SERVICES:
    svc_status, svc_data = gcp_request("GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}")
    if svc_status != 200:
        raise SystemExit(f"❌ HALT: Failed to query service '{svc}' state (HTTP {svc_status}): {svc_data}")
    state = svc_data.get("state", "DISABLED")
    if state != "ENABLED":
        disabled_apigee_svcs.append(svc)
    print(f"{svc:<40} | {state}")

if disabled_apigee_svcs:
    svc_str = " ".join(disabled_apigee_svcs)
    print("\n❌ Required Apigee API(s) disabled. Enable them explicitly with:\n")
    print(f"gcloud services enable {svc_str} --project={PROJECT_ID}")
    raise SystemExit("❌ HALT: Required Apigee API(s) are not enabled.")

PREFLIGHT["apigee_apis"] = True
print("\n✅ Apigee core APIs are ENABLED.")

In [ ]:
#@title 0.6 Verify Gemini / Vertex AI Readiness & Live Connectivity
# Verifies Vertex AI API enablement (gated by ENABLE_APIS), ensures aiplatform.endpoints.predict
# permission on the active caller (auto-binding roles/aiplatform.user when the caller holds
# resourcemanager.projects.setIamPolicy), and runs one live generateContent call.
import time

# 1. Check required Vertex AI / AI Gateway backend services
AI_SERVICES = [
    "aiplatform.googleapis.com",
]
disabled_ai_svcs = []
print(f"{'AI SERVICE':<40} | {'STATE'}")
print("-" * 55)
for svc in AI_SERVICES:
    svc_status, svc_data = gcp_request("GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}")
    if svc_status != 200:
        raise SystemExit(f"❌ HALT: Failed to query service '{svc}' state (HTTP {svc_status}): {svc_data}")
    state = svc_data.get("state", "DISABLED")
    if state != "ENABLED":
        disabled_ai_svcs.append(svc)
    print(f"{svc:<40} | {state}")

if disabled_ai_svcs:
    svc_str = " ".join(disabled_ai_svcs)
    enable_cmd = f"gcloud services enable {svc_str} --project={PROJECT_ID}"
    if ENABLE_APIS:
        print(f"\n⏳ ENABLE_APIS=True — running: {enable_cmd}")
        subprocess.run(["gcloud", "services", "enable", *disabled_ai_svcs, f"--project={PROJECT_ID}"], check=True)
        print("✅ Enabled required AI service(s).")
    else:
        print(f"\n❌ Disabled AI service(s) detected. Set ENABLE_APIS=True and re-run, or run manually:\n\n{enable_cmd}")
        raise SystemExit("❌ HALT: Vertex AI API is not enabled.")

# 2. Verify caller permission to invoke Gemini (auto-bind roles/aiplatform.user if caller can setIamPolicy)
VERTEX_PERM = "aiplatform.endpoints.predict"
VERTEX_ROLE = "roles/aiplatform.user"
member_prefix = "serviceAccount" if ACTIVE_IDENTITY.endswith(".gserviceaccount.com") else "user"
caller_member = f"{member_prefix}:{ACTIVE_IDENTITY}"
iam_test_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions"

v_status, v_iam = gcp_request(
    "POST",
    iam_test_url,
    {"permissions": [VERTEX_PERM, "resourcemanager.projects.setIamPolicy"]},
)
if v_status != 200:
    raise SystemExit(f"❌ HALT: Vertex AI testIamPermissions call failed (HTTP {v_status}): {v_iam}")

granted_vertex = set(v_iam.get("permissions", []))
if VERTEX_PERM not in granted_vertex:
    if "resourcemanager.projects.setIamPolicy" in granted_vertex:
        print(f"\n⏳ Granting '{VERTEX_ROLE}' to '{caller_member}' on project '{PROJECT_ID}'...")
        bind_res = subprocess.run(
            [
                "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
                f"--member={caller_member}",
                f"--role={VERTEX_ROLE}",
                "--condition=None",
                "--quiet",
            ],
            capture_output=True,
            text=True,
        )
        if bind_res.returncode != 0:
            raise SystemExit(
                f"❌ HALT: Failed to grant '{VERTEX_ROLE}' to '{caller_member}':\n"
                f"{(bind_res.stderr or bind_res.stdout).strip()}"
            )
        for _ in range(10):
            time.sleep(3)
            CREDENTIALS.refresh(Request())
            _, recheck = gcp_request("POST", iam_test_url, {"permissions": [VERTEX_PERM]})
            if VERTEX_PERM in recheck.get("permissions", []):
                granted_vertex.add(VERTEX_PERM)
                break

ok = VERTEX_PERM in granted_vertex
print(f"\nIAM Check: {VERTEX_PERM} -> {'PASS' if ok else 'FAIL'} (Role: {VERTEX_ROLE})")
if not ok:
    print(
        f"\n❌ Missing permission '{VERTEX_PERM}'. Grant it with:\n"
        f"gcloud projects add-iam-policy-binding {PROJECT_ID} \\\n"
        f"  --member=\"{caller_member}\" \\\n"
        f"  --role=\"{VERTEX_ROLE}\""
    )
    raise SystemExit("❌ HALT: Missing Vertex AI prediction permission.")

# 3. Make ONE minimal live generateContent call (retrying for IAM cache propagation and
# checking global / us-central1 fallback if the regional endpoint does not host GEMINI_MODEL)
payload = {
    "contents": [{"role": "user", "parts": [{"text": "Reply with the single word READY."}]}],
    "generationConfig": {"maxOutputTokens": 8, "temperature": 0.0},
}

def invoke_gemini(location: str, model_id: str):
    host = "aiplatform.googleapis.com" if location == "global" else f"{location}-aiplatform.googleapis.com"
    url = (
        f"https://{host}/v1/projects/{PROJECT_ID}/locations/{location}"
        f"/publishers/google/models/{model_id}:generateContent"
    )
    status, data = gcp_request("POST", url, payload, headers={"x-goog-user-project": PROJECT_ID})
    if status == 403:
        status, data = gcp_request("POST", url, payload)
    return status, data

VERTEX_LOCATION = REGION
gen_status, gen_data = 0, {}
fallback_locations = [loc for loc in ["global", "us-central1"] if loc != REGION]

for attempt in range(1, 13):
    CREDENTIALS.refresh(Request())
    gen_status, gen_data = invoke_gemini(REGION, GEMINI_MODEL)
    if gen_status == 200:
        VERTEX_LOCATION = REGION
        break

    # Vertex AI returns HTTP 403 "... (or it may not exist)" both during IAM cache propagation
    # and when a model ID (e.g. gemini-2.5-flash) is served on 'global'/'us-central1' rather than REGION.
    if gen_status in (403, 404):
        for fb_loc in fallback_locations:
            fb_status, fb_data = invoke_gemini(fb_loc, GEMINI_MODEL)
            if fb_status == 200:
                time.sleep(2)
                CREDENTIALS.refresh(Request())
                re_status, re_data = invoke_gemini(REGION, GEMINI_MODEL)
                if re_status == 200:
                    gen_status, gen_data = re_status, re_data
                    VERTEX_LOCATION = REGION
                else:
                    print(
                        f"ℹ️ Model '{GEMINI_MODEL}' is not served on regional endpoint '{REGION}' "
                        f"(HTTP {re_status}), but succeeded on Vertex AI location '{fb_loc}'."
                    )
                    gen_status, gen_data = fb_status, fb_data
                    VERTEX_LOCATION = fb_loc
                break
        if gen_status == 200:
            break

    if gen_status != 403:
        break

    print(f"⏳ Waiting for Vertex AI IAM cache propagation in '{REGION}' (attempt {attempt}/12)...")
    time.sleep(5)

if gen_status != 200:
    raise SystemExit(f"❌ HALT: Live Gemini call failed in '{REGION}' (HTTP {gen_status}): {gen_data}")

os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION
reply_text = (
    gen_data.get("candidates", [{}])[0]
    .get("content", {})
    .get("parts", [{}])[0]
    .get("text", "")
    .strip()
)
usage = gen_data.get("usageMetadata", {})
PREFLIGHT["gemini"] = True
print(f"✅ Model '{GEMINI_MODEL}' reachable in Vertex AI location '{VERTEX_LOCATION}'.")
print(f"✅ Live Gemini response: {reply_text!r} (promptTokens={usage.get('promptTokenCount')}, totalTokens={usage.get('totalTokenCount')})")

In [ ]:
#@title 0.7 Install & Verify AFT CLI
# Installs and verifies the Apigee Feature Templater ('aft') CLI.
# By default, 'aft' resolves templates and features directly from
# https://github.com/gcp-samples/apigee-template-repository and caches them in ~/.aft/cache/.
import pathlib
import shutil

os.environ.pop("AFT_REPOSITORY", None)

for extra_bin in [str(pathlib.Path.home() / ".aft" / "bin"), str(pathlib.Path.home() / ".local" / "bin")]:
    if extra_bin not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra_bin}:{os.environ['PATH']}"

if not shutil.which("aft"):
    subprocess.run(
        "curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh",
        shell=True,
        check=True,
    )

aft_bin = shutil.which("aft")
if not aft_bin:
    raise SystemExit("❌ HALT: 'aft' binary not found on PATH after installation.")

aft_ver = subprocess.run([aft_bin, "--version"], capture_output=True, text=True, check=True).stdout.strip()
PREFLIGHT["aft"] = True

print(f"✅ AFT CLI ready: {aft_bin} ({aft_ver})")

In [ ]:
#@title 0.8 Preflight Summary Gate
# Final blocking gate: verifies that all Step 0 checks succeeded before allowing execution
# of any downstream area.
failed_gates = [k for k, v in PREFLIGHT.items() if not v]
if failed_gates:
    raise SystemExit(f"❌ HALT: Preflight incomplete. Failed or unexecuted gate(s): {', '.join(failed_gates)}")

SUMMARY_ROWS = [
    ("GCP Project",        PROJECT_ID),
    ("Apigee Org",         f"{APIGEE_ORG} (ACTIVE)"),
    ("Apigee Env / Group", f"{APIGEE_ENV} -> {ENV_GROUP}"),
    ("Gateway Hostname",   HOSTNAME),
    ("Region",             REGION),
    ("Active Identity",    ACTIVE_IDENTITY),
    ("Apigee IAM Status",  "PASS (all effective permissions verified)"),
    ("Service Account",    f"{SERVICE_ACCOUNT} ({SA_STATUS})"),
    ("Apigee APIs Status", "PASS (apigee.googleapis.com, cloudresourcemanager.googleapis.com)"),
    ("Gemini Readiness",   f"PASS (model={GEMINI_MODEL}, location={globals().get('VERTEX_LOCATION', REGION)}, live call succeeded)"),
    ("AFT Toolkit",        f"READY ({aft_ver})"),
]

print("=" * 72)
for label, val in SUMMARY_ROWS:
    print(f"{label:<20}: {val}")
print("=" * 72)
print("PREFLIGHT PASSED — safe to proceed")

## Area 1 — LLM Gateway (Completions)

### 1.1 Use Case Summary — LLM Gateway
- **What we are deploying:** An Apigee AI Gateway abstraction layer that accepts standard OpenAI-compatible `/v1/chat/completions` payloads and routes requests to Google Cloud Vertex AI.
- **Why it matters:** Client applications remain decoupled from underlying backend model endpoints and provider-specific wire formats, sharing one consistent API contract with project and routing configuration driven by Step 0 (`PROJECT_ID`, `REGION`, and `SERVICE_ACCOUNT`).
- **Template & Scope:** Built from [`templates/llm-gateway-completions-no-auth.yaml`](https://raw.githubusercontent.com/luiscuellarh/apigee-unified-ai-gateway/refs/heads/main/templates/llm-gateway-completions-no-auth.yaml). Consumer-facing authentication is deliberately out of scope in this `-no-auth` variant and is introduced in **Area 4**.


In [ ]:
#@title 1.2 Fetch Template, Map Parameters, Prep Workspace
# Fetches templates/llm-gateway-completions-no-auth.yaml (and its referenced ../features/*.yaml)
# from the authoring repository and maps Step 0 variables to the template parameter contract.
import json
import pathlib
import urllib.request
import urllib.error

AUTHORING_REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway"
RAW_REPO_BASE = "https://raw.githubusercontent.com/luiscuellarh/apigee-unified-ai-gateway/refs/heads/main"
TEMPLATE_REL_PATH = "templates/llm-gateway-completions-no-auth.yaml"
TEMPLATE_RAW_URL = f"{RAW_REPO_BASE}/{TEMPLATE_REL_PATH}"

REFERENCED_FEATURES = [
    "features/ai-endpoint-completions.yaml",
    "features/ai-target-googlecloud.yaml",
    "features/ai-model-preprocess.yaml",
    "features/ai-model-failover.yaml",
    "features/ai-model-postprocess.yaml",
    "features/ai-token-quota.yaml",
]

WORKSPACE_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway-workspace"
(WORKSPACE_DIR / "templates").mkdir(parents=True, exist_ok=True)
(WORKSPACE_DIR / "features").mkdir(parents=True, exist_ok=True)

def fetch_repo_file(rel_path: str) -> pathlib.Path:
    url = f"{RAW_REPO_BASE}/{rel_path}"
    dest = WORKSPACE_DIR / rel_path
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "apigee-llm-gateway-colab"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            if resp.status != 200:
                raise SystemExit(f"❌ HALT: Failed to fetch '{url}' (HTTP {resp.status}).")
            content = resp.read().decode("utf-8")
    except urllib.error.URLError as exc:
        raise SystemExit(f"❌ HALT: Could not retrieve '{url}': {exc}")
    if not content.strip():
        raise SystemExit(f"❌ HALT: Retrieved empty file from '{url}'.")
    dest.write_text(content, encoding="utf-8")
    return dest

TEMPLATE_LOCAL_PATH = fetch_repo_file(TEMPLATE_REL_PATH)
for feat_rel in REFERENCED_FEATURES:
    fetch_repo_file(feat_rel)

os.environ["AFT_REPOSITORY"] = AUTHORING_REPO_URL

# Map Step 0 variables to the template parameter contract
AREA1_PARAMETERS = {
    "GoogleCloudProject": PROJECT_ID,
    "SimpleModel": "google/gemini-3.5-flash-lite",
    "MediumModel": "google/gemini-3.8-flash",
    "ComplexModel": "google/gemini-3.8-flash",
    "ThinkingModel": "anthropic/claude-opus-5",
    "ModelRouting": "'google/*'=googlecloud-oai,'zai-org/*=googlecloud-oai','anthropic/*'=googlecloud",
    "ModelMapping": f"gemini-flash-latest=gemini-3.7-flash,{GEMINI_MODEL}=gemini-3.7-flash",
}

print(f"✅ Template fetched: {TEMPLATE_RAW_URL}")
print(f"✅ Local path:       {TEMPLATE_LOCAL_PATH}")
print(f"✅ AFT_REPOSITORY:   {os.environ['AFT_REPOSITORY']}")
print("\n📋 Resolved Parameter Payload (Area 1 Contract):")
print(json.dumps(AREA1_PARAMETERS, indent=2))

In [ ]:
#@title 1.3 Render & Deploy via AFT
# Renders and deploys templates/llm-gateway-completions-no-auth.yaml to APIGEE_ENV using AFT.
import time

PROXY_NAME_LLM_ROUTING = os.environ.get("PROXY_NAME_LLM_ROUTING", "ai-model-routing-completions").strip() or "ai-model-routing-completions"
os.environ["PROXY_NAME_LLM_ROUTING"] = PROXY_NAME_LLM_ROUTING

CREDENTIALS.refresh(Request())

# Pass scalar contract parameters via --parameters; comma-delimited routing/mapping strings
# are already defined in the fetched template defaults.
cli_param_str = ",".join(
    f"{k}={AREA1_PARAMETERS[k]}"
    for k in ("GoogleCloudProject", "SimpleModel", "MediumModel", "ComplexModel", "ThinkingModel")
)

aft_cmd = [
    aft_bin,
    "convert",
    "-i", str(TEMPLATE_LOCAL_PATH),
    "-n", PROXY_NAME_LLM_ROUTING,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENV,
    "--service-account", SERVICE_ACCOUNT,
    "--parameters", cli_param_str,
    "--token", CREDENTIALS.token,
    "--no-animation",
]

print(f"⏳ Rendering and deploying proxy '{PROXY_NAME_LLM_ROUTING}' to env '{APIGEE_ENV}' via AFT...")
deploy_res = subprocess.run(aft_cmd, cwd=str(WORKSPACE_DIR), capture_output=True, text=True)
if deploy_res.returncode != 0:
    err_out = (deploy_res.stderr or deploy_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT deployment failed (exit {deploy_res.returncode}):\n{err_out}")

# Poll Apigee environment deployments until the proxy revision reports READY
dep_url = f"{APIGEE_BASE}/environments/{APIGEE_ENV}/apis/{PROXY_NAME_LLM_ROUTING}/deployments"
deployed_rev, dep_state = None, "UNKNOWN"
for _ in range(24):
    d_status, d_data = gcp_request("GET", dep_url)
    deployments = d_data.get("deployments", []) if d_status == 200 else []
    if deployments:
        latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
        deployed_rev = latest.get("revision")
        dep_state = latest.get("state", "READY")
        if dep_state == "READY":
            break
    time.sleep(5)

if dep_state != "READY":
    raise SystemExit(
        f"❌ HALT: Proxy '{PROXY_NAME_LLM_ROUTING}' did not reach READY state in env '{APIGEE_ENV}' "
        f"(revision={deployed_rev}, state={dep_state})."
    )

LLM_COMPLETIONS_ENDPOINT = f"https://{HOSTNAME}/v1/chat/completions"
os.environ["LLM_COMPLETIONS_ENDPOINT"] = LLM_COMPLETIONS_ENDPOINT

print(f"✅ Proxy Name:        {PROXY_NAME_LLM_ROUTING}")
print(f"✅ Deployed Revision: {deployed_rev} (state={dep_state})")
print(f"✅ Endpoint URL:      {LLM_COMPLETIONS_ENDPOINT}")

In [ ]:
#@title 1.4 Validation & Live Test
# Executes three live calls against the deployed /v1/chat/completions endpoint:
# 1) gemini-3.7-flash (expect HTTP 200)
# 2) gemini-3.1-pro   (expect HTTP 200 and model switch)
# 3) missing model    (expect HTTP 4xx rejection from OAS validation)
import time
import requests

def diagnose_gateway_failure(status_code: int, body) -> str:
    text = str(body).lower()
    if status_code in (401, 403) or "permission_denied" in text or "aiplatform.endpoints.predict" in text:
        return (
            f"Likely cause: Missing IAM permission or API disabled. Verify that service account "
            f"'{SERVICE_ACCOUNT}' holds 'roles/aiplatform.user' and 'aiplatform.googleapis.com' is ENABLED in '{PROJECT_ID}'."
        )
    if status_code == 404 or "not found" in text or "does not exist" in text:
        return (
            f"Likely cause: Requested model is not available on the Vertex AI endpoint in project '{PROJECT_ID}' "
            f"(region='{REGION}' / global) or the model ID is not published in Vertex AI Model Garden."
        )
    if status_code == 429 or "quota" in text or "resource_exhausted" in text:
        return f"Likely cause: Vertex AI quota or rate limit exhausted in project '{PROJECT_ID}'."
    return f"Likely cause: Upstream Vertex AI or gateway routing error (HTTP {status_code})."

def call_completions_endpoint(payload: dict):
    t0 = time.perf_counter()
    try:
        resp = requests.post(
            LLM_COMPLETIONS_ENDPOINT,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=45,
        )
    except requests.RequestException as exc:
        raise SystemExit(
            f"❌ HALT: Network/TLS error calling '{LLM_COMPLETIONS_ENDPOINT}': {exc}\n"
            f"Verify that load balancer IP/DNS and managed SSL certificate for '{HOSTNAME}' are ACTIVE."
        )
    latency_ms = (time.perf_counter() - t0) * 1000.0
    try:
        body = resp.json()
    except Exception:
        body = {"raw": resp.text}
    return resp.status_code, latency_ms, body

# Test 1: model = "gemini-3.7-flash"
status_1, latency_1, body_1 = call_completions_endpoint({
    "model": "gemini-3.7-flash",
    "messages": [{"role": "user", "content": "Reply with one short sentence confirming gemini-3.7-flash."}],
    "max_tokens": 32,
    "temperature": 0.0,
})
if status_1 != 200:
    print(f"❌ Test 1 (gemini-3.7-flash) returned HTTP {status_1}: {body_1}")
    print(f"🔎 {diagnose_gateway_failure(status_1, body_1)}")
    raise SystemExit("❌ HALT: Test 1 failed (expected HTTP 200).")

model_used_1 = body_1.get("model", "gemini-3.7-flash")
snippet_1 = (
    body_1.get("choices", [{}])[0]
    .get("message", {})
    .get("content", "")
    .strip()
    .replace("\n", " ")[:120]
)
print(f"✅ Test 1 PASS (HTTP {status_1}) | model={model_used_1} | latency={latency_1:.0f}ms | response={snippet_1!r}")

# Test 2: model = "gemini-3.1-pro"
status_2, latency_2, body_2 = call_completions_endpoint({
    "model": "gemini-3.1-pro",
    "messages": [{"role": "user", "content": "Reply with one short sentence confirming gemini-3.1-pro."}],
    "max_tokens": 32,
    "temperature": 0.0,
})
if status_2 != 200:
    print(f"❌ Test 2 (gemini-3.1-pro) returned HTTP {status_2}: {body_2}")
    print(f"🔎 {diagnose_gateway_failure(status_2, body_2)}")
    raise SystemExit("❌ HALT: Test 2 failed (expected HTTP 200).")

model_used_2 = body_2.get("model", "gemini-3.1-pro")
snippet_2 = (
    body_2.get("choices", [{}])[0]
    .get("message", {})
    .get("content", "")
    .strip()
    .replace("\n", " ")[:120]
)
if model_used_2 == model_used_1:
    raise SystemExit(f"❌ HALT: Expected model to switch from '{model_used_1}', but response still reported '{model_used_2}'.")
print(f"✅ Test 2 PASS (HTTP {status_2}) | model={model_used_2} (switched from {model_used_1}) | latency={latency_2:.0f}ms | response={snippet_2!r}")

# Test 3: Omit "model" — assert clean 4xx rejection
status_3, latency_3, body_3 = call_completions_endpoint({
    "messages": [{"role": "user", "content": "This request omits the required model field."}],
})
if not (400 <= status_3 < 500):
    raise SystemExit(f"❌ HALT: Test 3 (missing model) expected HTTP 4xx rejection, but got HTTP {status_3}: {body_3}")
print(f"✅ Test 3 PASS (HTTP {status_3}) | Missing 'model' cleanly rejected as expected ({latency_3:.0f}ms)")

## Area 2 — Prompt-Injection and Jailbreak Guardrails
- **Purpose:** Apply Google Cloud Model Armor screening to inspect incoming prompts and outgoing LLM completions for prompt-injection, jailbreak, and safety violations before returning responses.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/ai-security-modelarmor.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)

In [ ]:
#@title Area 2 — Prompt-Injection and Jailbreak Guardrails (Placeholder)
# TODO: Verify Model Armor API/template readiness, compose features/ai-security-modelarmor.yaml onto the AI proxy via aft, and test benign vs. prompt-injection payloads.
pass

## Area 3 — DLP / PII Detection and Redaction
- **Purpose:** De-identify and mask sensitive PII (such as emails, phone numbers, and credit card numbers) in user prompts before forwarding payloads to upstream LLM providers.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/security-anonymization-sdp.yaml`, `features/security-anonymization-presidio.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)

In [ ]:
#@title Area 3 — DLP / PII Detection and Redaction (Placeholder)
# TODO: Verify Cloud Sensitive Data Protection (dlp.googleapis.com) readiness, compose features/security-anonymization-sdp.yaml via aft, and test synthetic PII redaction.
pass

## Area 4 — Authentication and Authorization
- **Purpose:** Enforce client authentication and consumer group authorization on AI Gateway endpoints using Apigee API keys, OAuth 2.1 tokens, and KVM-backed group lookups while stripping client credentials before target invocation.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/auth-apikey-verify.yaml`, `features/auth-group-lookup.yaml`, `features/auth-oauth-verify.yaml`, `features/auth-oauth-server.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-Completions.yaml`, `templates/REST-AI-GenerateContent.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)

In [ ]:
#@title Area 4 — Authentication and Authorization (Placeholder)
# TODO: Provision an Apigee API Product, Developer, and Developer App, apply features/auth-apikey-verify.yaml and features/auth-group-lookup.yaml via aft, and test authorized vs. unauthorized requests.
pass

## Area 5 — Token Quota, Rate Limiting, Spike Arrest
- **Purpose:** Enforce LLM token consumption quotas per API Product/App (`total_tokens` weighted quota) using the published Stable Feature, while noting that Spike Arrest and KVM request rate-limiting exist only in Drafts.
- **Stable Template / Feature Citations:**
  - **Token Quota (`## Stable Features`):** `features/ai-token-quota.yaml` (pre-integrated in `templates/REST-AI-Completions.yaml`)
  - **Spike Arrest & Dynamic KVM Request Rate Limiting:** **`[NO STABLE OPTION]`** — `features/draft/spike-arrest-burst-smoother.yaml` and `features/draft/rate-limit-dynamic-kvm.yaml` sit under `## Draft Extension Features` and are excluded; only `features/ai-token-quota.yaml` is in Stable.
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing), Area 4 (Authentication and Authorization)

In [ ]:
#@title Area 5 — Token Quota, Rate Limiting, Spike Arrest (Placeholder)
# TODO: Configure token_quota, token_quota_interval, and token_quota_timeunit on the Apigee API Product and verify HTTP 200 within budget vs. HTTP 429 on token quota exhaustion.
pass

## Area 6 — Token Counting and Cost Attribution
- **Purpose:** Extract input, output, reasoning, and cached token counts from both non-streaming and SSE streaming LLM responses and calculate per-request cost using the `PriceList` entry in the `AI-Config` KVM.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/ai-model-postprocess.yaml`, `features/ai-model-preprocess.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)

In [ ]:
#@title Area 6 — Token Counting and Cost Attribution (Placeholder)
# TODO: Provision Apigee Data Collectors (dc_ai_model, dc_ai_provider, dc_input_token_count, dc_output_token_count, dc_cached_token_count, dc_reasoning_token_count, dc_total_token_count, dc_total_token_cost), seed PriceList in AI-Config KVM, and verify token/cost calculation.
pass

## Area 7 — Caching `[NO STABLE OPTION]`
- **Purpose:** Document the stability assessment for LLM response and semantic caching against the base repository.
- **Stable Template / Feature Citations:**
  - **`[NO STABLE OPTION]`:** Neither `## Stable Templates` nor `## Stable Features` includes an HTTP response cache or semantic cache policy (`features/draft/response-cache-invalidation.yaml` sits under `## Draft Extension Features` and is excluded).
  - **Closest Stable Templates as Base Proxy (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml` (note: `features/ai-model-postprocess.yaml` tracks upstream provider `cached_token_count` billing metrics, not gateway-layer response caching).
- **Dependencies:** Step 0 (Preflight)

In [ ]:
#@title Area 7 — Caching [NO STABLE OPTION] (Placeholder)
# TODO: No-op or await explicit instruction on how to handle Area 7 given [NO STABLE OPTION] in gcp-samples/apigee-template-repository.
pass

## Area 8 — Model Allow-Listing and Failover
- **Purpose:** Enforce per-product model allow-lists (`allowed_models` with wildcard/prefix matching) and automatic target failover to a backup Vertex AI model (`FailoverModel`) on upstream faults.
- **Stable Template / Feature Citations:**
  - **Model Allow-Listing (`## Stable Features`):** `features/ai-token-quota.yaml` (`JS-ValidateRequest` and `AM-ModelNotAllowed`)
  - **Model Failover (`## Stable Features`):** `features/ai-model-failover.yaml`, `features/ai-model-preprocess.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-Completions.yaml`, `templates/REST-AI-GenerateContent.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing), Area 4 (Authentication and Authorization)

In [ ]:
#@title Area 8 — Model Allow-Listing and Failover (Placeholder)
# TODO: Configure allowed_models on the API Product and FailoverModel in the AI-Config KVM, compose features/ai-model-failover.yaml via aft, and test HTTP 400 model rejection and automatic failover.
pass

## Area 9 — Logging and Observability
- **Purpose:** Capture AI token, cost, model, and provider telemetry into Apigee Analytics via `DC-TokenAnalytics` and provision custom analytics reports, while noting external log/trace sinks exist only in Drafts.
- **Stable Template / Feature Citations:**
  - **Apigee AI Analytics & Data Collectors (`## Stable Features`):** `features/ai-model-postprocess.yaml` (`DC-TokenAnalytics` and `JS-Analytics`)
  - **External Log & Trace Sinks:** **`[NO STABLE OPTION]`** — `features/draft/cloud-logging-sink.yaml`, `features/draft/opentelemetry-trace-exporter.yaml`, `features/draft/datadog-metrics-forwarder.yaml`, and `features/draft/splunk-hec-forwarder.yaml` sit under `## Draft Extension Features` and are excluded.
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing), Area 6 (Token Counting and Cost Attribution)

In [ ]:
#@title Area 9 — Logging and Observability (Placeholder)
# TODO: Provision the AI Custom Analytics Reports (AI Token Cost by App & Model, AI Model Latency by Provider, AI Token Count by Consumer Group) and query Apigee Analytics dimensions.
pass

## Area 10 — Summary + Optional Cleanup
- **Purpose:** Print an inventory of all Apigee AI Gateway resources created during the session and optionally tear them down behind an explicit `CONFIRM_DELETE = False` `#@param` guard.
- **Stable Template / Feature Citations:** Notebook lifecycle utility operating on resources deployed from `## Stable Templates` and `## Stable Features`.
- **Dependencies:** Step 0 (Preflight), Areas 1–9

In [ ]:
#@title Area 10 — Summary + Optional Cleanup (Placeholder)
CONFIRM_DELETE = False #@param {type:"boolean"}
# TODO: Print deployed proxy/product/app/KVM/collector inventory and, only when CONFIRM_DELETE is True, undeploy and delete session resources cleanly.
pass